# 05 · Online vs offline — a live Red Hat AI MaaS endpoint

Notebooks 00–04 run **offline**: the agent is canned and the verdict comes from recorded fixtures. This notebook adds the **online** option — point the same probe and gate at a live **Red Hat AI MaaS** (Models-as-a-Service) model.

MaaS endpoints are OpenAI-compatible and gated by an API key (issued/rate-limited by 3scale). When you turn it on, the OWASP probe stops *reading a file* and becomes a **real jailbreak attempt against a real model** — and the rc1→rc2 difference becomes a genuine behavioral change, not a scripted one.

**The safety net is unconditional:** every online path degrades to the offline path on any error, missing config, or timeout. Online is pure upside — it can never break the demo.

In [ ]:
import os
from pathlib import Path

# Notebooks live in <repo>/notebooks; walk up to the dir that holds the Makefile.
root = Path.cwd()
while not (root / "Makefile").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working dir:", Path.cwd())

## Going online — supply the endpoint + key via `.env`
Credentials come from a **gitignored** `.env` at the repo root (never committed). Copy the template and fill it in:

```bash
cp .env.example .env       # then edit: MAAS_ENDPOINT, MAAS_MODEL, MAAS_API_KEY
```

> Provision a **dedicated demo key** the morning of the talk — one small model, low rate limit, spend cap, expiry — and revoke it right after. Never paste the key on a shared screen; the harness only ever shows its last 4 characters.

### Check what mode you're in (without printing the key)

In [ ]:
from harness import maas

cfg = maas.maas_config()
if cfg:
    print("ONLINE — live Red Hat MaaS")
    print("  endpoint:", cfg["endpoint"])
    print("  model:   ", cfg["model"])
    print("  key:     ", maas.masked_key(), "(last 4 only)")
else:
    print("OFFLINE — no .env / MAAS_* set; canned + fixtures.")
    print("  Copy .env.example to .env and fill it in to go online.")

## The OWASP probe — offline vs online
Both always produce a verdict. Offline is the canned deterministic path; online sends the system prompt + the adversarial message to the live model and inspects what it *actually* generates (degrading to canned if MaaS is unset/unreachable).

**Offline** (canned, no network):

In [ ]:
!make probe

**Online** (live MaaS; note the `[probe via …]` provenance line tells you which path answered):

In [ ]:
!make probe-live

> On **rc1** a live model with no confidentiality instruction may well leak the prompt; after `make harden` (rc2), the hardened prompt makes it refuse **and** the output guardrail blocks any residual leak. Run `03-harden-and-promote.ipynb`'s harden/unharden around `make probe-live` to see the live flip.

## The release gate — offline vs online
The gate has the same two options. `auto` (the default, `make gate`) picks online if EvalHub is reachable and falls back to offline otherwise — so it's safe on stage.

**Offline** (fixtures only, instant):

In [ ]:
!make gate-offline; echo; echo "(offline verdict above)"

**Online** (real submit + poll against EvalHub → MaaS). Needs `make up` and a `.env`, and runs the full suite (minutes, not seconds), so it's left commented — uncomment when you have EvalHub up and a reachable endpoint:

In [ ]:
# !make up            # start EvalHub + MLflow (one time)
# !make gate-live     # real online run; degrades to fixtures on any failure

### How online targets MaaS
When `.env` is set, `apply_maas_override()` rewrites the job's `model` to your MaaS endpoint at submit time (the committed spec stays secret-free), and EvalHub reads the key from the `maas-api-key` secret — `docker-compose.yaml` passes `MAAS_API_KEY` through for you. Offline and replay modes never touch `.model`, so they're unaffected.

## Safety recap
- The key lives only in `.env` (**gitignored**) — never in the spec, notebook, or repo.
- It's never printed: logs show only the last 4 characters.
- Notebooks are committed without output, so a stray value can't be saved.
- Every online path degrades to offline, so a dead key / rate-limit / wifi never breaks the demo.
- Use a short-lived, scoped, capped demo key and revoke it after the talk.